# Notebook 12: VLA Action Diffusion (Toy)

**目标**：在玩具 2D 环境上实现 diffusion policy。

**任务**：smile 形状轨迹追踪——从随机 init 走到目标点的连续 trajectory，用 diffusion 一次性生成。

**前置**：L17, Diffusion Policy paper

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. 构造 "专家 demonstration" 数据

每个 demonstration = (起始位置 obs, 一段 32-step trajectory)。
trajectory 是从 obs 走到目标点的平滑路径。

In [ ]:
def gen_demo(n=2000, H=32, target=(1.0, 0.0)):
    """生成 n 条从随机起点到 target 的平滑 trajectory"""
    target = torch.tensor(target)
    obs_list = []
    traj_list = []
    for _ in range(n):
        # 起点：随机 in [-1, 1]^2
        start = torch.rand(2) * 2 - 1
        obs_list.append(start)
        # 平滑 trajectory: minimum jerk (近似 sigmoid 加速)
        t = torch.linspace(0, 1, H + 1)
        # smoothstep: 6t^5 - 15t^4 + 10t^3
        s = 6*t**5 - 15*t**4 + 10*t**3
        pos = start.unsqueeze(0) + s.unsqueeze(-1) * (target - start).unsqueeze(0)
        # action = position differences
        actions = pos[1:] - pos[:-1]
        # 加点小噪声让数据更真实
        actions = actions + 0.005 * torch.randn_like(actions)
        traj_list.append(actions)
    return torch.stack(obs_list), torch.stack(traj_list)

obs, traj = gen_demo(2000)
print(f'obs: {obs.shape}, traj: {traj.shape}')
print(f'traj per step: mean={traj.abs().mean():.3f}, std={traj.std():.3f}')

# 可视化几条 demo trajectory
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
for i in range(20):
    pos = obs[i].unsqueeze(0) + traj[i].cumsum(0)
    pos = torch.cat([obs[i].unsqueeze(0), pos], dim=0)
    ax.plot(pos[:, 0], pos[:, 1], alpha=0.5)
    ax.scatter(obs[i, 0], obs[i, 1], c='green', s=30, marker='o')
ax.scatter(1.0, 0.0, c='red', s=100, marker='*', label='target')
ax.set_aspect('equal'); ax.legend(); ax.set_title('Expert demonstrations')
plt.show()

## 2. Diffusion Policy: action chunk diffusion

Model 输入：(obs, t, noisy_action_chunk)
输出：noise prediction

Action chunk 32 × 2 = 64 维。把它当作 "image" 来 diffuse。

In [ ]:
class DiffusionPolicy(nn.Module):
    def __init__(self, obs_dim=2, action_dim=2, horizon=32, hidden=256):
        super().__init__()
        self.horizon = horizon
        self.action_dim = action_dim
        flat_action_dim = horizon * action_dim
        # 时间 embedding
        self.t_emb = nn.Sequential(nn.Linear(1, 64), nn.SiLU(), nn.Linear(64, 64))
        # Obs conditioning
        self.obs_proj = nn.Sequential(nn.Linear(obs_dim, 64), nn.SiLU(), nn.Linear(64, 64))
        # 主网络：MLP 处理 (noisy_action, t_emb, obs_emb)
        self.net = nn.Sequential(
            nn.Linear(flat_action_dim + 128, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, flat_action_dim),
        )
    def forward(self, action_noisy, t, obs):
        # action_noisy: (B, H, A)
        B = action_noisy.shape[0]
        a_flat = action_noisy.reshape(B, -1)
        t_e = self.t_emb(t.float().unsqueeze(-1) / 1000.0)
        o_e = self.obs_proj(obs)
        x = torch.cat([a_flat, t_e, o_e], dim=-1)
        eps_flat = self.net(x)
        return eps_flat.reshape(B, self.horizon, self.action_dim)

model = DiffusionPolicy().to(device)
print(f'params: {sum(p.numel() for p in model.parameters())/1e3:.1f}K')

## 3. DDPM 训练

In [ ]:
T_diff = 100  # 小 T 即可
betas = torch.linspace(1e-4, 0.02, T_diff).to(device)
alphas = 1 - betas
ac = alphas.cumprod(0)

obs = obs.to(device); traj = traj.to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(3000):
    idx = torch.randint(0, len(obs), (256,))
    o = obs[idx]
    a = traj[idx]
    t = torch.randint(0, T_diff, (a.shape[0],), device=device)
    eps = torch.randn_like(a)
    a_noisy = ac[t].view(-1,1,1).sqrt() * a + (1-ac[t]).view(-1,1,1).sqrt() * eps
    loss = F.mse_loss(model(a_noisy, t, o), eps)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        print(f'step {step}: loss={loss.item():.4f}')

## 4. 推理：从新起点生成 action chunk

In [ ]:
@torch.no_grad()
def sample_action(obs_in, n_steps=20):
    model.eval()
    a = torch.randn(obs_in.shape[0], 32, 2, device=device)
    step = T_diff // n_steps
    for ti in reversed(range(0, T_diff, step)):
        t = torch.full((obs_in.shape[0],), ti, device=device, dtype=torch.long)
        eps = model(a, t, obs_in)
        a0 = (a - (1-ac[ti]).sqrt() * eps) / ac[ti].sqrt()
        a0 = a0.clamp(-1, 1)
        if ti - step >= 0:
            a = ac[ti-step].sqrt() * a0 + (1 - ac[ti-step]).sqrt() * eps
        else:
            a = a0
    return a

# 测试：5 个不同起点
torch.manual_seed(7)
test_obs = (torch.rand(5, 2) * 2 - 1).to(device)
actions = sample_action(test_obs, n_steps=20)

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
for i in range(5):
    o = test_obs[i].cpu()
    a = actions[i].cpu()
    pos = torch.cat([o.unsqueeze(0), o.unsqueeze(0) + a.cumsum(0)], dim=0)
    ax.plot(pos[:, 0], pos[:, 1], '-o', markersize=3, alpha=0.7, label=f'demo {i}')
    ax.scatter(o[0], o[1], c='green', s=80, marker='o', edgecolor='black')
ax.scatter(1.0, 0.0, c='red', s=150, marker='*', label='target')
ax.set_aspect('equal'); ax.legend(loc='upper left'); ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_title('Diffusion Policy: generated trajectories from random starts')
plt.show()

## 5. 多模态动作分布

扩散 policy 的核心优势：能处理多模态分布。

实验：把数据集改成有**两个 target**（左右各一个），同一起点应当生成两种 trajectory。

In [ ]:
# 重新生成 dataset，target 是 (1, 0) 或 (-1, 0) 随机选
obs2_list, traj2_list = [], []
for _ in range(2000):
    start = torch.rand(2) * 2 - 1
    target = torch.tensor([1.0, 0.0]) if torch.rand(1).item() < 0.5 else torch.tensor([-1.0, 0.0])
    t = torch.linspace(0, 1, 33)
    s = 6*t**5 - 15*t**4 + 10*t**3
    pos = start.unsqueeze(0) + s.unsqueeze(-1) * (target - start).unsqueeze(0)
    actions = pos[1:] - pos[:-1] + 0.005 * torch.randn(32, 2)
    obs2_list.append(start); traj2_list.append(actions)
obs2 = torch.stack(obs2_list).to(device); traj2 = torch.stack(traj2_list).to(device)

# 重训
model2 = DiffusionPolicy().to(device)
opt = torch.optim.Adam(model2.parameters(), lr=1e-3)
for step in range(3000):
    idx = torch.randint(0, len(obs2), (256,))
    o, a = obs2[idx], traj2[idx]
    t = torch.randint(0, T_diff, (a.shape[0],), device=device)
    eps = torch.randn_like(a)
    a_noisy = ac[t].view(-1,1,1).sqrt() * a + (1-ac[t]).view(-1,1,1).sqrt() * eps
    loss = F.mse_loss(model2(a_noisy, t, o), eps)
    opt.zero_grad(); loss.backward(); opt.step()

# 在同一起点反复采样
@torch.no_grad()
def sample_with_model2(obs_in, n_steps=20):
    model2.eval()
    a = torch.randn(obs_in.shape[0], 32, 2, device=device)
    step = T_diff // n_steps
    for ti in reversed(range(0, T_diff, step)):
        t = torch.full((obs_in.shape[0],), ti, device=device, dtype=torch.long)
        eps = model2(a, t, obs_in)
        a0 = (a - (1-ac[ti]).sqrt() * eps) / ac[ti].sqrt()
        a0 = a0.clamp(-1, 1)
        if ti - step >= 0:
            a = ac[ti-step].sqrt() * a0 + (1 - ac[ti-step]).sqrt() * eps
        else:
            a = a0
    return a

torch.manual_seed(0)
fixed_start = torch.tensor([[0.0, 0.5]]).repeat(20, 1).to(device)
actions = sample_with_model2(fixed_start)

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
for i in range(20):
    o = fixed_start[i].cpu(); a = actions[i].cpu()
    pos = torch.cat([o.unsqueeze(0), o.unsqueeze(0) + a.cumsum(0)], dim=0)
    ax.plot(pos[:, 0], pos[:, 1], '-', alpha=0.5)
ax.scatter(0.0, 0.5, c='green', s=150, marker='o', label='start (fixed)')
ax.scatter(1.0, 0.0, c='red', s=150, marker='*', label='target 1')
ax.scatter(-1.0, 0.0, c='blue', s=150, marker='*', label='target 2')
ax.set_aspect('equal'); ax.legend()
ax.set_title('20 trajectories from same start → 两种 target 模式应当被识别')
plt.show()

## 观察

如果模型工作正常，20 条 trajectory 应当大致分成 2 组（向左 / 向右）——这正是**多模态分布**。
用 behavior cloning (MSE on action) 会**只学到平均**（直接朝下，撞墙）—— 这就是 diffusion policy 的优势。

## 思考题

1. 实现 action chunking + 推理只用前 K 步 模式（K=10），多次重新规划
2. 把 obs 维度从 2 改成 8（如加上目标位置作为 obs 一部分），看条件性增强
3. 把网络换成 Transformer 或 1D-UNet（更类似 Diffusion Policy paper）
4. 用 Flow Matching 替代 DDPM 实现 action 生成，看速度/质量对比